In [ ]:
import os
import pandas as pd
import random

current_folder = os.getcwd()
project_folder = os.path.dirname(current_folder)

csv_file = os.path.join(project_folder, "Data", "EloRatings_clean.csv")

df = pd.read_csv(csv_file)

clubs = list(sorted(df["club"].unique()))
countries = sorted(df["country"].unique())

clubs_country = df.set_index("club")["country"].to_dict()

sorted_clubs_country = dict(sorted(clubs_country.items(), key = lambda item: item[1]))

df_1750 = (df[(df["elo"] > 1750) & (df["country"] != "RUS")]
    .sort_values(by="elo", ascending=False)
    .drop_duplicates(subset="club"))

Selected_teams = random.sample(df_1750["club"].tolist(), 36)

print(Selected_teams)
print(clubs_country)

def Newlistwithoutteam(team):
    Listcopy = Selected_teams.copy()
    Listcopy.remove(team)
    return Listcopy

def country_code():
    Club_code = {}
    for team in Selected_teams:
        Club_code[team] = clubs_country[team]
    return Club_code

country_code()





def is_valid_game(home_team, away_team, teams):
    game = (home_team, away_team)
    game_switch = (away_team, home_team)
    Club_code = country_code()
    if game in teams[home_team]["Matches"]:
        return False
    elif game in teams[away_team]["Matches"]:
        return False
    elif home_team == away_team:
        return False
    elif game_switch in teams[home_team]["Matches"]:
        return False
    elif game_switch in teams[away_team]["Matches"]:
        return False 
    elif Club_code[home_team] == Club_code[away_team]:
        return False 
    elif len(teams[home_team]["Matches"]) > 7:
        return False
    elif len(teams[away_team]["Matches"]) > 7:
        return False
    else:
        return True 

def initialize_games():
    teams = {}
    for team in Selected_teams:
        teams[team] = {"Matches": [] }
    return teams 

def generate_all_games():
    all_games = []
    for teamA in Selected_teams:
        for teamB in Selected_teams:
            if teamA != teamB:
                home_team = teamA
                away_team = teamB
                game = (home_team, away_team)
                all_games.append(game)
    return all_games
    

def shuffle_games():
    all_games = generate_all_games()
    random.shuffle(all_games)
    return all_games

def valid_games():
    while True: 
        teams = initialize_games()
        all_games = shuffle_games()
        for game in all_games:
            home_team, away_team = game
            if is_valid_game(home_team, away_team, teams): 
                teams[home_team]["Matches"].append(game) 
                teams[away_team]["Matches"].append(game)
                if all_teams_complete(teams):
                    return teams
                 
        

def all_teams_complete(teams):
    for team in teams:
        if len(teams[team]["Matches"]) < 8:
            return False
    return True
            



def print_table(teams):
    print(f"{'Team':<25} {'Wedstrijden'}")
    print("-" * 70)

    for team in teams:
        print(f"{team:<25} {len(teams[team]['Matches'])}  {teams[team]['Matches']}")

print(Selected_teams)
teams = valid_games()
print_table(teams)
print(len(Selected_teams))          # moet 36 zijn
print(len(set(Selected_teams)))  


['Sochaux', 'Celta', 'Reading', 'RB Leipzig', 'Brentford', 'Sunderland', 'Levante', 'Aston Villa', 'Stuttgart', 'Porto', 'Espanol', 'Newcastle', 'Brighton', 'Hertha', 'Man City', 'Girona', 'Ath Bilbao', 'Santander', 'Betis', 'Valencia', 'West Ham', 'Fulham', 'Chelsea', 'Tottenham', 'Osasuna', 'Union Berlin', 'Juventus', 'Valladolid', 'Feyenoord', 'Parma', 'Nottm Forest', 'Leicester', 'Leverkusen', 'Bournemouth', 'Arsenal', 'Sp Lisbon']
{'A. Sebatspor': 'TUR', 'AEK': 'GRE', 'AIK': 'SWE', 'AVS': 'POR', 'AZ Alkmaar': 'NED', 'Aachen': 'GER', 'Aalborg': 'DEN', 'Aalen': 'GER', 'Aalesund': 'NOR', 'Aalst': 'BEL', 'Aarhus': 'DEN', 'Aberdeen': 'SCO', 'Academica': 'POR', 'Ad. Demirspor': 'TUR', 'Adanaspor': 'TUR', 'Admira': 'AUT', 'Ahlen': 'GER', 'Ajaccio': 'FRA', 'Ajax': 'NED', 'Akademisk': 'DEN', 'Akhisar Belediyespor': 'TUR', 'Akratitos': 'GRE', 'Akron Togliatti': 'RUS', 'Alanya': 'RUS', 'Alanyaspor': 'TUR', 'Alaves': 'ESP', 'Albacete': 'ESP', 'Albinoleffe': 'ITA', 'Alcorcon': 'ESP', 'Alcoyano

In [ ]:
import math
import numpy as np 
import pandas as pd
import os
current_folder = os.getcwd()
project_folder = os.path.dirname(current_folder)

csv_file1 = os.path.join(project_folder, "data", "Matches_clean.csv")

df1 = pd.read_csv(csv_file1)

## Import data
teams = sorted(pd.concat([df1["HomeTeam"], df1["AwayTeam"]]).unique())
Goals_scored = 0
teams_data = {}
Goals_conceded = 0
Totalgoals = float(df1["FTHome"].sum() + df1["FTAway"].sum())
TotalAverage = float(Totalgoals / len(df1))
Info = {}

def Goalsscored(team):
    homegoals = float(df1.loc[df1["HomeTeam"] == team, "FTHome"].sum())
    awaygoals = float(df1.loc[df1["AwayTeam"] == team, "FTAway"].sum())
    return homegoals + awaygoals


def Goalsconceded(team):
    homeconceded = float(df1.loc[df1["HomeTeam"] == team, "FTAway"].sum())
    awayconceded = float(df1.loc[df1["AwayTeam"] == team, "FTHome"].sum())
    return homeconceded + awayconceded

def Matches_played(team):
    return float(len(df1[(df1["HomeTeam"] == team) | (df1["AwayTeam"] == team)]))


def Rating(team):   
    GS = Goalsscored(team)
    GC = Goalsconceded(team)
    GS_pergame = (GS)/Matches_played(team)
    GC_pergame = (GC)/Matches_played(team)
    Attack = GS_pergame / TotalAverage
    Defense = GC_pergame / TotalAverage
    Info[team] = {"Scored": GS, "Scored_pergame": GS_pergame, "Conceded": GC, "Conceded_pergame": GC_pergame, "Attack_rating": Attack, "Defense_rating": Defense}
    print(Info[team])

Rating("Barcelona")
Rating("Feyenoord")

    
def Parameter(team1, team2):
    AttackTeam1 = ratings[team1]["Attack"]
    DefenseTeam1 =  ratings[team1]["Defense"]
    AttackTeam2 = ratings[team2]["Attack"]
    DefenseTeam2 =  ratings[team2]["Defense"]
    ParameterTeam1 = TotalAverage * AttackTeam1 * DefenseTeam2
    ParameterTeam2 = TotalAverage * AttackTeam2 * DefenseTeam1
    return ParameterTeam1, ParameterTeam2
    

{'Scored': 867.0, 'Scored_pergame': 2.381868131868132, 'Conceded': 333.0, 'Conceded_pergame': 0.9148351648351648, 'Attack_rating': 0.8943471745727583, 'Defense_rating': 0.34350358608157844}
